### Chat Models in LangChain

Chat models are language models that use a sequence of messages as inputs and return messages as outputs (as opposed to traditional, plaintext LLMs).

Unlike traditional text-completion LLMs operating on raw strings, modern chat models process structured multi-role conversations.

---

### Core Capabilities of Chat Models

* Structured Message Processing: Handles multi-role inputs (SystemMessage, HumanMessage, AIMessage, ToolMessage).
* Tool & Function Calling: Binds external Python functions via model.bind_tools().
* Structured Output Parsing: Enforces JSON/Pydantic schemas using model.with_structured_output().
* Streaming & Batching: Supports token-by-token streaming (model.stream()) and parallel batch processing (model.batch()).
* Multimodal Support: Processes text, images, and audio payloads within messages.

### Chat Model Method Matrix

| Method / Concept | Description | Typical Usage |
| :--- | :--- | :--- |
| init_chat_model() | Unified initializer for instantiating model providers | init_chat_model("gemini-3.6-flash", model_provider="google_genai") |
| model.invoke() | Synchronous call returning a complete AIMessage | response = model.invoke(messages) |
| model.stream() | Iterative generator yielding message chunks for streaming UI | for chunk in model.stream(messages): print(chunk.content) |
| model.batch() | Executes multiple prompt inputs concurrently | responses = model.batch([messages1, messages2]) |
| model.bind_tools() | Attaches tools/functions to the model for tool calling | model_with_tools = model.bind_tools([my_tool]) |
| model.with_structured_output() | Wraps model to return validated Pydantic objects | structured_llm = model.with_structured_output(MySchema) |

### Popular Chat Model Provider Integrations

LangChain provides first-party integration packages for all major LLM providers:

| Provider | Integration Class | Package Name | Environment Key |
| :--- | :--- | :--- | :--- |
| Google Gemini | ChatGoogleGenerativeAI | langchain-google-genai | GOOGLE_API_KEY |
| OpenAI | ChatOpenAI | langchain-openai | OPENAI_API_KEY |
| Anthropic | ChatAnthropic | langchain-anthropic | ANTHROPIC_API_KEY |
| Azure OpenAI | AzureChatOpenAI | langchain-openai | AZURE_OPENAI_API_KEY |
| Groq | ChatGroq | langchain-groq | GROQ_API_KEY |
| HuggingFace | ChatHuggingFace | langchain-huggingface | HUGGINGFACEHUB_API_TOKEN |
| Ollama (Local) | ChatOllama | langchain-ollama | Localhost (No Key Needed) |
| xAI (Grok) | ChatXAI | langchain-xai | XAI_API_KEY |
| NVIDIA | ChatNVIDIA | langchain-nvidia-ai-endpoints | NVIDIA_API_KEY |

---

### Official Documentation Reference

For full provider installation instructions and API reference, see: https://docs.langchain.com/oss/python/integrations/chat

### 1. Model Initialization

Using init_chat_model() with Google AI Studio (model_provider="google_genai"):

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()


### Chat Model Parameters

When initializing a chat model, you can configure hyper-parameters to control randomness, response length, sampling behavior, and network resiliency:

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| model | str | Required | Model checkpoint name (e.g. gemini-3.6-flash, gpt-4o). |
| temperature | float | 0.7 | Controls output randomness (0.0 = deterministic/focused, 1.0 = creative). |
| max_tokens | int | None | Maximum number of tokens the model can generate in output. |
| top_p | float | 1.0 | Nucleus sampling: considers tokens with cumulative probability top_p. |
| top_k | int | None | Limits candidate tokens to the top_k most probable words. |
| timeout | float | None | Max time in seconds to wait for an API response. |
| max_retries | int | 2 | Number of automatic retries on rate-limit or network errors. |
| stop | list[str] | None | Sequences that stop model text generation when encountered. |

Example configuring parameters with init_chat_model():
model = init_chat_model("gemini-3.6-flash", model_provider="google_genai", temperature=0.2, max_tokens=1000)

In [ ]:
model = init_chat_model("gemini-3.6-flash", model_provider="google_genai")

### 2. LLM Output

We can use IPython.display.display and Markdown for printing output beautifully:

In [ ]:
# from utils import print_md
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a senior Python developer."),
    HumanMessage(content="Explain REST API design with a short example.")
]

# Generate response from LLM
response = model.invoke(messages)

# print_md(response.content)
print(response.content)

### 3. Provider Class Import Examples

Reference examples for instantiating provider classes directly:

In [ ]:
# Google Gemini
from langchain_google_genai import ChatGoogleGenerativeAI
# llm_google = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

# OpenAI
# from langchain_openai import ChatOpenAI
# llm_openai = ChatOpenAI(model="gpt-4o")

# Anthropic
# from langchain_anthropic import ChatAnthropic
# llm_anthropic = ChatAnthropic(model="claude-3-5-sonnet-20241022")

# Azure OpenAI
# from langchain_openai import AzureChatOpenAI
# llm_azure = AzureChatOpenAI(azure_deployment="your-deployment", api_version="2024-02-01")

# Groq
# from langchain_groq import ChatGroq
# llm_groq = ChatGroq(model="llama-3.3-70b-versatile")

# HuggingFace
# from langchain_huggingface import ChatHuggingFace
# llm_hf = ChatHuggingFace(repo_id="mistralai/Mistral-7B-Instruct-v0.2")

# Ollama (Local)
# from langchain_ollama import ChatOllama
# llm_ollama = ChatOllama(model="llama3")

# xAI (Grok)
# from langchain_xai import ChatXAI
# llm_xai = ChatXAI(model="grok-beta")

# NVIDIA
# from langchain_nvidia_ai_endpoints import ChatNVIDIA
# llm_nvidia = ChatNVIDIA(model="meta/llama-3.1-70b-instruct")

### 4. Streaming LLM Responses (model.stream)

Streaming allows language models to emit tokens iteratively as they are generated, rather than blocking until the full completion response is constructed. This dramatically reduces Time-To-First-Token (TTFT) and delivers responsive real-time user experiences.

---

### Core Concepts in Streaming

| Concept / Attribute | Description | Code Pattern |
| :--- | :--- | :--- |
| model.stream(input) | Synchronous iterator yielding token chunks (AIMessageChunk) as they arrive. | for chunk in model.stream(input): |
| model.astream(input) | Async generator for streaming tokens inside async event loops. | async for chunk in model.astream(input): |
| chunk.content / chunk.text | Extracts the raw string fragment from an individual message chunk. | print(chunk.text, end="", flush=True) |
| Chunk Addition (+) | Merges message chunks sequentially (chunk1 + chunk2), automatically combining text content, token usage metadata, and partial tool call JSON payloads. | accumulated += chunk |

---

### Streaming Output Patterns

1. Raw Terminal / Console Stream: Using print(chunk.text, end="", flush=True) to stream plain text directly to the console output stream.
2. Live Markdown Rendering (Jupyter UI): Using IPython.display's display(Markdown(...), display_id=True) to dynamically update rendered Markdown in-place in notebook cells.
3. Chunk Concatenation & Accumulation: Using the + operator on AIMessageChunk instances to assemble the final complete AIMessage object.

In [ ]:
# 1. Raw Text Token Streaming with custom chunk delimiter '|'
print("--- Streaming Raw Chunks ---")
for chunk in model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

### Chunk Accumulation with + Operator

LangChain AIMessageChunk objects implement python addition (+), allowing you to combine chunks seamlessly into a single complete response:

In [28]:
# 3. Accumulating AIMessageChunks using `+` operator
full_chunk = None

for chunk in model.stream("Give a 1-sentence tip for writing clean Python code."):
    if full_chunk is None:
        full_chunk = chunk
    else:
        full_chunk += chunk

print(f"Accumulated object type: {type(full_chunk).__name__}")
print(f"Final combined content:\n{full_chunk.content}")

Accumulated object type: AIMessageChunk
Final combined content:
[{'type': 'text', 'text': 'Prioritize readability by adhering to PEP 8 standards and using clear, descriptive names for functions and variables so your code is self-documenting.', 'index': 0, 'extras': {'signature': 'EosRCogRARFNMg/4pN2KB610C9+qqOZo4mLV/v8G5BASVsJtrrhpwlq3tvNm1IQz8/IbAKomeGqusvhGmyIAMCdKJT+TeIpplB09whU5Hxo5NcEeucDskKZj7weWyelFvcRFN+NnO2Nwj8a9pNkuRplLhgLcItnPNcQ5e1V3esdxF2X0zQcoEFMt8C/CLmJtpDMynRoInUiDXlOLwxnfmCq20BIxhS9P4/jghcxV/L7TECaIc+mcbBVfoKZ9p2k7YXG+BWEAhFF3VrnDy9lMiODA/+wnSta2RaWZTdizwHfSOgzLjFPz+qhlivm+lGHLvCwqLzgCzzA20q8yhqH4420DQR2NL3VMheZmtCCtWUxADBOy96QSECWQPYey4qlov5TxTL2PmpKdZbbH201MXZ3GA5FQhCU8ra6lgM1Z7uCWtHBpSNKTa2LANb6vCO8oT7QypgnZbw7smNwGX4UiQ9LAWOg/qsg23QSE68ccgGl9CJByQfpSgcAtZaezTjiCZ2H1THTTVpg6o0NGQLqLKQ5HIfAeR/sBVsfcQJYneE+06b2vutuc3a2ZXt1VqiwXME00sh1aunqxpqIaKt2ZuWLsmka8ZUkzrCfAdMsOxTeAysrR9XZD8tAjR9+6yNWjIUEq4VkfKBW5vXnDYdhCj8b6dzGeLYr/RJEp0XFYl023WqegLPwQBe1Y0wg/i0X9qiw6+YeJ/RgW68M

### 5. Batch Processing (model.batch)

Batching allows processing multiple prompt inputs concurrently rather than sequentially one by one. This reduces total processing time when dealing with multiple queries or datasets.

---

### Key Concepts in Batching

| Concept / Method | Description | Usage Pattern |
| :--- | :--- | :--- |
| model.batch(inputs) | Executes a list of inputs concurrently and returns a list of AIMessage outputs. | responses = model.batch(prompts) |
| model.abatch(inputs) | Async version for executing concurrent batches within async event loops. | responses = await model.abatch(prompts) |
| max_concurrency config | Limits the maximum number of parallel API requests to prevent rate limit errors. | model.batch(prompts, config={"max_concurrency": 3}) |

---

### Batch Output Characteristics

1. Preserved Ordering: The returned list of AIMessage objects matches the exact order of the input prompts list.
2. Error Handling: Failed requests in a batch can be managed with retry configurations or fallback handlers.
3. Concurrency Limits: Passing max_concurrency in config ensures API rate limits are respected when executing large batches.

In [ ]:
prompts = [
    "Explain Python in 1 short sentence.",
    "Explain JavaScript in 1 short sentence.",
    "Explain Rust in 1 short sentence."
]

responses = model.batch(prompts)

for prompt, response in zip(prompts, responses):
    print(f"Prompt: {prompt}")
    print(f"Response: {response.content}\n")

### Batching Message Lists with Concurrency Control

You can batch structured message lists and control the parallel execution rate using the max_concurrency configuration:

In [ ]:
# 2. Batching structured message lists with concurrency limit
from langchain_core.messages import SystemMessage, HumanMessage

batch_messages = [
    [
        SystemMessage(content="You are a translation assistant."),
        HumanMessage(content="Translate 'Hello, how are you?' into Spanish.")
    ],
    [
        SystemMessage(content="You are a translation assistant."),
        HumanMessage(content="Translate 'Hello, how are you?' into French.")
    ],
    [
        SystemMessage(content="You are a translation assistant."),
        HumanMessage(content="Translate 'Hello, how are you?' into German.")
    ]
]

# Set max_concurrency to 2 to limit parallel API calls
responses = model.batch(batch_messages, config={"max_concurrency": 2})

for i, res in enumerate(responses, 1):
    print(f"Translation {i}: {res.content}")